# Xây Dựng Và Đánh Giá Chatbot Nhà Hàng Dựa Trên RAG

**Đồ án:** Restaurant QR AI Ordering  
**Phương pháp:** Retrieval-Augmented Generation (RAG) với BM25, Dense E5, Hybrid RRF  
**Ngôn ngữ:** Python 3.13 · PyTorch · Sentence-Transformers

---

Notebook này trình bày **quá trình xây dựng từ đầu**, chạy code thật tại mỗi bước.  
Mọi con số được tính trực tiếp — không dùng dữ liệu tĩnh hay hình ảnh sẵn có.

In [ ]:
import sys, os
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["font.size"] = 11

AI_ROOT = Path(".").resolve()
if AI_ROOT.name == "notebooks":
    AI_ROOT = AI_ROOT.parent
KB_PATH = AI_ROOT / "knowledge-base"
PROJECT_ROOT = AI_ROOT.parent

sys.path.insert(0, str(AI_ROOT))
os.chdir(AI_ROOT)

print(f"AI_ROOT:  {AI_ROOT}")
print(f"KB_PATH:  {KB_PATH}")
print(f"Python:   {sys.version.split()[0]}")

---
# PHẦN I — BÀI TOÁN VÀ DỮ LIỆU

## 1. Bài toán

Khách hàng quét mã QR tại bàn và **chat với AI** để hỏi về menu, giá, chính sách,
dị ứng, gợi ý món. Hệ thống phải trả lời chính xác dựa trên dữ liệu thật.

### 3 ràng buộc bất khả xâm phạm

| Ràng buộc | Ý nghĩa | Cách đảm bảo |
|---|---|---|
| **Grounding** | Không bịa món, không bịa giá | Đối chiếu menu thật trước khi trả lời |
| **Confirmation** | Không tự đặt món thay khách | Guardrail → frontend hiện nút xác nhận |
| **Safety** | Không tiết lộ PII / system prompt | Regex + guardrails chặn trước LLM |

### Đầu ra không chỉ là văn bản

Hệ thống trả về **structured response** gồm: text, decision, evidence IDs,
claims, cart actions, session updates — để backend kiểm tra lại.

### Kiến trúc evidence-first

| Loại câu hỏi | Ví dụ | Nguồn dữ liệu | Cần LLM? |
|---|---|---|---|
| Chào hỏi, cảm ơn | "Xin chào" | Deterministic | Không |
| Giá / còn bán | "Phở bò bao nhiêu?" | Live database (menu API) | Có |
| FAQ / chính sách | "Wifi pass là gì?" | Knowledge Base → RAG | Có |

> **Quyết định:** Evidence-first routing giảm latency, tránh hallucination về giá.

## 2. Khám phá Knowledge Base

Knowledge Base (KB) chứa **tri thức tĩnh** của nhà hàng: FAQ, chính sách, thông tin
dị ứng, combo, thanh toán... dưới dạng file Markdown.

> **Nguồn dữ liệu:** `knowledge-base/` — 26 file Markdown, load bằng `load_markdown_knowledge_base()`

In [ ]:
from app.rag.knowledge_base import load_markdown_knowledge_base

kb_chunks = load_markdown_knowledge_base(KB_PATH)
print(f"Tổng số chunks: {len(kb_chunks)}")
print(f"Số file nguồn:  {len(set(c.source for c in kb_chunks))}")
print(f"Tất cả có tags: {all(c.tags for c in kb_chunks)}")

### 2.1 Phân bố chunk theo file nguồn

In [ ]:
source_counts = Counter(c.source for c in kb_chunks)
source_df = pd.DataFrame([
    {"Tệp nguồn": src, "Số chunk": cnt}
    for src, cnt in source_counts.most_common()
])
display(source_df.style.hide(axis="index"))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(source_df["Tệp nguồn"][::-1], source_df["Số chunk"][::-1], color="#0ea5e9")
ax.set_xlabel("Số chunk")
ax.set_title("Phân bố chunk theo file nguồn")
plt.tight_layout()
plt.show()

### 2.2 Phân bố theo mức độ rủi ro (risk tier)

Mỗi chunk được gán risk tier dựa trên **hậu quả nếu AI trả lời sai**:

| Risk tier | Tiêu chí | Ví dụ |
|---|---|---|
| **critical** | Sai → nguy hại sức khỏe | Dị ứng, cross-contamination |
| **high** | Sai → mất tiền | Chính sách hoàn tiền, thanh toán |
| **medium** | Sai → trải nghiệm kém | FAQ, combo gợi ý |
| **low** | Sai → không ảnh hưởng | Lịch sử nhà hàng, brand voice |

In [ ]:
tier_counts = Counter(c.risk_tier for c in kb_chunks)
tier_df = pd.DataFrame([
    {"Risk tier": tier, "Số chunk": cnt}
    for tier, cnt in sorted(tier_counts.items(), key=lambda x: ["critical","high","medium","low"].index(x[0]))
])
display(tier_df.style.hide(axis="index"))

colors = {"critical": "#ef4444", "high": "#f97316", "medium": "#eab308", "low": "#22c55e"}
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(tier_df["Risk tier"], tier_df["Số chunk"], color=[colors[t] for t in tier_df["Risk tier"]])
ax.set_ylabel("Số chunk")
ax.set_title("Phân bố chunk theo risk tier")
plt.tight_layout()
plt.show()

### 2.3 Question variants — làm giàu index cho BM25

BM25 chỉ match **từ chính xác**. Nếu heading là *"Tiện Nghi"* mà khách hỏi
*"wifi pass"* → không có từ chung → miss.

**Giải pháp:** Thêm `<!-- question_variants: wifi, pass wifi -->`
ngay sau heading. Khi chunking, variants được nối vào content → BM25 có thêm từ khóa.

In [ ]:
import re

variant_files = []
no_variant_files = []
for f in sorted(KB_PATH.glob("*.md")):
    text = f.read_text(encoding="utf-8")
    count = len(re.findall(r"question_variants", text))
    if count > 0:
        variant_files.append({"Tệp": f.name, "Số variant line": count})
    else:
        no_variant_files.append(f.name)

print(f"Files CÓ question variants:  {len(variant_files)}/26")
print(f"Files KHÔNG cần variants:    {len(no_variant_files)}/26")
print()
display(pd.DataFrame(variant_files).style.hide(axis="index").set_caption("Files CÓ question variants"))

#### Minh họa cơ chế: Variants giúp BM25 vượt vocabulary mismatch

3 queries mà khách dùng từ khác heading KB:

In [ ]:
from app.rag.retriever import BM25Retriever
from app.rag.knowledge_base import KnowledgeChunk
import re as _re

bm25_with = BM25Retriever(kb_chunks)

stripped_chunks = [KnowledgeChunk(
    source=c.source, title=c.title,
    content=_re.sub(r"question_variants:.*", "", c.content),
    tags=c.tags, chunk_id=c.chunk_id, risk_tier=c.risk_tier,
) for c in kb_chunks]
bm25_without = BM25Retriever(stripped_chunks)

demo_queries = [
    ("có chỗ đậu xe không?",  "restaurant-info.md",    "Heading 'Gửi Xe' — khách nói 'đậu xe'"),
    ("món nào không cay?",     "spice-flavor-scale.md", "Heading 'Thang Cay' — khách nói 'không cay'"),
    ("mức cay mấy?",           "spice-flavor-scale.md", "Heading 'Thang Cay' — khách nói 'mức cay'"),
]

rows = []
for query, expected, explanation in demo_queries:
    rw = bm25_with.search(query, top_k=1)
    rwo = bm25_without.search(query, top_k=1)
    hit_w = rw[0].chunk.source if rw else "(miss)"
    hit_wo = rwo[0].chunk.source if rwo else "(miss)"
    ok_w = "✓" if rw and expected in hit_w else "✗"
    ok_wo = "✓" if rwo and expected in hit_wo else "✗"
    rows.append({"Query": query, "Vocabulary mismatch": explanation,
        "CÓ variants": f"{ok_w} {hit_w}", "KHÔNG variants": f"{ok_wo} {hit_wo}"})
display(pd.DataFrame(rows).style.hide(axis="index").set_caption(
    "Minh họa: variants giúp BM25 vượt vocabulary mismatch"))

Variants là kỹ thuật **document expansion** — thêm từ khóa vào document để BM25
có thể match. Impact sẽ được đo lường chính thức trong Phần II.

### 2.4 Chiến lược chunking

Mỗi file Markdown được chia thành chunks theo **heading cấp 2 (`##`)**.

| Chiến lược | Ưu điểm | Nhược điểm |
|---|---|---|
| Theo paragraph | Chunk nhỏ, cụ thể | Quá ngắn → thiếu ngữ cảnh |
| Theo file | Đầy đủ ngữ cảnh | Quá dài → retrieval kém |
| **Theo heading cấp 2** | Cân bằng | Chunk size không đều |

In [ ]:
import numpy as np

word_counts = [len(c.content.split()) for c in kb_chunks]
char_counts = [len(c.content) for c in kb_chunks]

stats = pd.DataFrame([{
    "Metric": "Số từ / chunk",
    "Min": min(word_counts), "Max": max(word_counts),
    "Trung bình": f"{np.mean(word_counts):.0f}",
    "Trung vị": f"{np.median(word_counts):.0f}",
}, {
    "Metric": "Số ký tự / chunk",
    "Min": min(char_counts), "Max": max(char_counts),
    "Trung bình": f"{np.mean(char_counts):.0f}",
    "Trung vị": f"{np.median(char_counts):.0f}",
}])
display(stats.style.hide(axis="index").set_caption("Thống kê kích thước chunk"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(word_counts, bins=20, color="#06b6d4", edgecolor="white")
ax.axvline(np.median(word_counts), color="#ef4444", linestyle="--", label=f"Median = {np.median(word_counts):.0f}")
ax.set_xlabel("Số từ")
ax.set_ylabel("Số chunk")
ax.set_title(f"Phân bố kích thước {len(kb_chunks)} chunks")
ax.legend()
plt.tight_layout()
plt.show()

### 2.5 Ví dụ chunk thật

In [ ]:
samples = {}
for c in kb_chunks:
    if c.risk_tier not in samples:
        samples[c.risk_tier] = c
    if len(samples) >= 3:
        break

for tier, chunk in samples.items():
    words = len(chunk.content.split())
    print(f"{'='*60}")
    print(f"Risk tier : {tier}")
    print(f"Source    : {chunk.source}")
    print(f"Title     : {chunk.title}")
    print(f"Kích thước: {words} từ, {len(chunk.content)} ký tự")
    print(f"Tags      : {', '.join(chunk.tags[:5])}")
    print(f"Content   : {chunk.content[:200]}...")
    print()

## 3. Chuẩn hóa tiếng Việt

Tiếng Việt tự nhiên có nhiều biến thể cần chuẩn hóa **trước khi** truy xuất:
- **Viết tắt**: "ko" → "không", "dc" → "được", "bn" → "bao nhiêu"
- **Không dấu**: "hai san" → "hải sản"
- **Emoji**: 🌶️ → "cay", 🦐 → "tôm"
- **Gen-Z**: "hông" → "không", "ntn" → "như thế nào"

### 3.1 Demo chuẩn hóa — bảng before / after

In [ ]:
from app.rag.vietnamese_normalizer import normalize_query_text, normalize_vietnamese

test_cases = [
    "ko co mon nao ngon ko?",
    "hai san tuoi khong?",
    "budget 500k cho 3 nguoi",
    "dc ko, bn tien?",
    "PHAI KHONG????",
    "pho bo bao nhieu tien",
]

norm_rows = []
for q in test_cases:
    norm_rows.append({
        "Câu gốc": q,
        "normalize_query_text()": normalize_query_text(q),
        "normalize_vietnamese()": normalize_vietnamese(q),
    })

display(pd.DataFrame(norm_rows).style.hide(axis="index"))

### 3.2 Hai hàm normalize — mục đích khác nhau

| Hàm | Mục đích | Dùng ở đâu |
|---|---|---|
| `normalize_query_text()` | **BM25 matching** — strip dấu, teencode, emoji | Built-in BM25 tokenizer |
| `normalize_vietnamese()` | **NLU** — giữ dấu, chỉ sửa teencode | Intent detection, LLM prompt |

`normalize_query_text()` được gọi **bên trong** BM25 tokenizer —
cả lúc index documents lẫn lúc search. Impact của normalize sẽ được
đo lường chính thức trong Phần II.

## 4. Tập đánh giá retrieval

Để so sánh các phương pháp retrieval **công bằng**, cần tập dữ liệu
với nhãn chính xác.

### Phương pháp xây dựng

1. **Family-based design**: Mỗi intent có 1 template, sinh 5 query variants.
2. **Curated templates**: Viết tay, dựa trên log chat thật.
3. **Engineering review**: Gán `expected_selectors` và `forbidden_selectors`.
4. **Frozen test split**: `dev` (thử nghiệm) và `test` (bị khóa, chỉ dùng cuối).
5. **Augmented cases**: 53 noisy + 19 vocab-mismatch để test preprocessing.

In [ ]:
import json

eval_path = AI_ROOT / "evaluation" / "datasets" / "retrieval_cases.dev.v1.jsonl"
eval_cases = [json.loads(line) for line in open(eval_path, "r", encoding="utf-8")]

original = [c for c in eval_cases if not c.get("noise_type")]
noisy = [c for c in eval_cases if c.get("noise_type") in ("no-diacritics", "teencode+no-diac")]
vocab = [c for c in eval_cases if c.get("noise_type") == "vocab-mismatch"]

print(f"Tổng số cases:     {len(eval_cases)}")
print(f"  Original:        {len(original)}")
print(f"  Noisy augmented: {len(noisy)}")
print(f"  Vocab mismatch:  {len(vocab)}")
print(f"Positive cases:    {sum(1 for c in eval_cases if c.get('expected_selectors'))}")
print(f"Negative cases:    {sum(1 for c in eval_cases if not c.get('expected_selectors'))}")

### 4.1 Phân bố theo intent

In [ ]:
intent_counts = Counter(c["intent"] for c in eval_cases)
intent_df = pd.DataFrame([
    {"Intent": intent, "Số case": count, "Tỷ lệ": f"{count/len(eval_cases):.0%}"}
    for intent, count in intent_counts.most_common()
])
display(intent_df.style.hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(intent_df["Intent"][::-1], intent_df["Số case"][::-1], color="#8b5cf6")
ax.set_xlabel("Số case")
ax.set_title(f"Phân bố {len(eval_cases)} eval cases theo intent")
plt.tight_layout()
plt.show()

### 4.2 Ví dụ eval case

In [ ]:
seen_intents = set()
example_rows = []
for c in eval_cases:
    if c["intent"] not in seen_intents and len(example_rows) < 5:
        seen_intents.add(c["intent"])
        example_rows.append({
            "Case ID": c["case_id"],
            "Intent": c["intent"],
            "Query": c["query"][:50],
            "Expected": str(c.get("expected_selectors", []))[:40],
        })
display(pd.DataFrame(example_rows).style.hide(axis="index"))

### 4.3 Negative cases — out_of_catalog

15 cases hỏi về thứ nhà hàng **không có** (gluten-free, keto, halal).
Retrieval **không nên** tìm thấy document phù hợp.

In [ ]:
negative_cases = [c for c in eval_cases if not c.get("expected_selectors")]
neg_df = pd.DataFrame([{
    "Case ID": c["case_id"],
    "Query": c["query"][:50],
    "Guardrail": ", ".join(c.get("guardrail_flags", [])),
} for c in negative_cases[:6]])
display(neg_df.style.hide(axis="index"))
print(f"Tổng negative cases: {len(negative_cases)}")

---
## Kết luận Phần I

### Lưu ý về dữ liệu

Đây là dự án tự xây dựng — KB, eval set, và các phương pháp đều do nhóm phát triển.
Kết quả chỉ phản ánh hiệu quả trên **bộ dữ liệu cụ thể này** (26 files KB, 197 eval cases),
không generalize ra các domain khác.

### Đã chuẩn bị

| Thành phần | Chi tiết |
|---|---|
| **Knowledge Base** | 26 files Markdown → 213 chunks, 4 risk tiers |
| **Preprocessing** | Normalize (teencode + không dấu + emoji), Question variants (14/26 files) |
| **Eval set** | 197 cases: 125 original + 53 noisy + 19 vocab-mismatch + 15 negative |

### Câu hỏi Phần II sẽ trả lời

Với dữ liệu đã chuẩn bị, Phần II sẽ xây dựng và so sánh **3 phương pháp retrieval**:

1. **BM25** (lexical) — match từ chính xác, có normalize built-in
2. **Dense E5** (semantic) — encode thành vector, so sánh theo nghĩa
3. **Hybrid RRF** — kết hợp BM25 + Dense bằng Reciprocal Rank Fusion

> Phương pháp nào tốt nhất trên bộ dữ liệu tự xây của chúng tôi?
> Normalize và variants thực sự cải thiện bao nhiêu?
> Và cái giá phải trả (latency, complexity) có xứng đáng không?

# PHẦN II — SO SÁNH CÁC PHƯƠNG PHÁP RETRIEVAL

## 5. Ba phương pháp retrieval

| Phương pháp | Loại | Cơ chế | Ưu điểm | Nhược điểm |
|---|---|---|---|---|
| **BM25** | Lexical | Khớp từ chính xác (TF-IDF) | Nhanh, không cần GPU | Không hiểu nghĩa |
| **Dense E5** | Semantic | Cosine similarity trên embedding | Hiểu paraphrase, đa ngôn ngữ | Cần load model |
| **Hybrid RRF** | Kết hợp | Reciprocal Rank Fusion | Bù nhược điểm cho nhau | Phức tạp nhất |

### 5.1 BM25 (Lexical)

Okapi BM25: TF-IDF + document length normalization.
Tokenizer có `normalize_query_text()` built-in + question variants nối vào chunk.

In [ ]:
import time
from app.rag.retriever import BM25Retriever

t0 = time.perf_counter()
bm25 = BM25Retriever(kb_chunks)
print(f"BM25 index: {(time.perf_counter()-t0)*1000:.0f}ms, {len(kb_chunks)} chunks")
for r in bm25.search("nhà hàng có wifi không?", top_k=3):
    print(f"  {r.score:.2f}  {r.chunk.source}  [{r.chunk.title}]")

### 5.2 Dense E5 (Semantic)

Trong notebook và báo cáo, **Dense E5** là tên phương pháp; trên hệ thống cùng một encoder được cấu hình bằng alias **`e5_small`** (model HuggingFace `intfloat/multilingual-e5-small`, 120MB, 384d).
Prefix: `"query: "` / `"passage: "`. Cosine similarity = dot product (normalized).

In [ ]:
from app.rag.embedding_retriever import DenseRetriever, create_encoder

t0 = time.perf_counter()
encoder = create_encoder("e5_small")
print(f"Encoder load: {(time.perf_counter()-t0)*1000:.0f}ms")

t0 = time.perf_counter()
dense = DenseRetriever(kb_chunks, encoder)
print(f"Doc encoding: {(time.perf_counter()-t0)*1000:.0f}ms")
print(f"Model: {encoder.model_name}, dim={encoder.dimension}")

for r in dense.search("nhà hàng có wifi không?", top_k=3):
    print(f"  {r.score:.4f}  {r.chunk.source}  [{r.chunk.title}]")

### 5.3 Hybrid RRF

$$RRF(d) = \sum_{r} \frac{w_r}{k + rank_r(d)}$$

`k=60`, `w=1.0` cho cả BM25 và Dense.

In [ ]:
from app.rag.hybrid_retriever import HybridRrfRetriever

hybrid = HybridRrfRetriever([bm25, dense])
print("Hybrid RRF: BM25 + Dense E5")
for r in hybrid.search("nhà hàng có wifi không?", top_k=3):
    print(f"  {r.score:.6f}  {r.chunk.source}  [{r.chunk.title}]")

## 6. Đánh giá trên eval set

### Scope và Metric

- **107 KB-relevant cases** (35 clean + 53 noisy + 19 vocab-mismatch)
- Không tính menu cases (category/tag) vì KB retriever không xử lý
- **Hit@1**: document đúng là kết quả đầu tiên? (ưu tiên production)
- **Hit@5**: document đúng nằm trong top-5?

> **Lưu ý thống kê:** Với n=19 (vocab mismatch), sai 1 case = thay đổi ~5%.
> Các khác biệt nhỏ trên tập này có thể do ngẫu nhiên, không nên kết luận quá mạnh.

> **Nguồn dữ liệu:** `evaluation/datasets/retrieval_cases.dev.v1.jsonl` (107 KB cases)
> Retriever: `BM25Retriever` + `DenseE5Retriever` + `HybridRrfRetriever` từ Part I KB

In [ ]:
import json

eval_cases = [json.loads(l) for l in open(
    AI_ROOT / "evaluation" / "datasets" / "retrieval_cases.dev.v1.jsonl",
    "r", encoding="utf-8"
)]

clean_kb = [c for c in eval_cases if c.get("expected_selectors")
    and any(s.startswith("kb-source") for s in c["expected_selectors"])
    and not c.get("noise_type")]
noisy = [c for c in eval_cases if c.get("noise_type") in ("no-diacritics", "teencode+no-diac")]
vocab = [c for c in eval_cases if c.get("noise_type") == "vocab-mismatch"]
negative = [c for c in eval_cases if not c.get("expected_selectors")]
kb_cases = clean_kb + noisy + vocab

print(f"KB cases: {len(kb_cases)} (clean={len(clean_kb)}, noisy={len(noisy)}, vocab={len(vocab)})")
print(f"Negative: {len(negative)}")

### 6.1 Kết quả tổng hợp (Hit@1 và Hit@5)

In [ ]:
def eval_hits(retriever, cases, top_k=5):
    hit1 = hit5 = 0
    for c in cases:
        if not c.get("expected_selectors"): continue
        results = retriever.search(c["query"], top_k=top_k)
        sources = [r.chunk.source for r in results]
        matched = lambda srcs: any(
            any(sel.split(":")[-1] in s for s in srcs)
            for sel in c["expected_selectors"]
        )
        if matched(sources[:1]): hit1 += 1
        if matched(sources[:5]): hit5 += 1
    return hit1, hit5

methods = {"BM25": bm25, "Dense E5": dense, "Hybrid RRF": hybrid}
groups = [("Clean KB", clean_kb), ("Noisy", noisy), ("Vocab mismatch", vocab), ("Tổng KB", kb_cases)]

rows = []
chart_data = {}
for gname, gcases in groups:
    n = len(gcases)
    row = {"Tập": f"{gname} ({n})"}
    for mname, ret in methods.items():
        h1, h5 = eval_hits(ret, gcases)
        row[f"{mname} @1"] = f"{h1}/{n} ({h1/n:.0%})"
        row[f"{mname} @5"] = f"{h5}/{n} ({h5/n:.0%})"
        chart_data[(gname, mname)] = h5 / n
    rows.append(row)

display(pd.DataFrame(rows).style.hide(axis="index").set_caption(
    "Hit@1 và Hit@5: 3 phương pháp retrieval"))

In [ ]:
import numpy as np

gnames = ["Clean KB", "Noisy", "Vocab mismatch", "Tổng KB"]
mnames = ["BM25", "Dense E5", "Hybrid RRF"]
colors = {"BM25": "#3b82f6", "Dense E5": "#8b5cf6", "Hybrid RRF": "#10b981"}

x = np.arange(len(gnames))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, m in enumerate(mnames):
    vals = [chart_data[(g, m)] for g in gnames]
    bars = ax.bar(x + i*width, vals, width, label=m, color=colors[m])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01,
                f"{val:.0%}", ha="center", va="bottom", fontsize=9)

ax.set_ylabel("Hit@5")
ax.set_title("So sánh 3 phương pháp retrieval (bộ dữ liệu tự xây)")
ax.set_xticks(x + width)
ax.set_xticklabels(gnames)
ax.set_ylim(0, 1.15)
ax.legend()
ax.axhline(y=1.0, color="#e5e7eb", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

### 6.2 False Positive — Negative cases

15 cases hỏi về thứ nhà hàng **không có** (gluten-free, keto, halal).
Retriever **không nên** trả về document nào có score cao.

False positive nguy hiểm vì AI sẽ dựa vào document sai để trả lời —
khách hàng nghĩ nhà hàng có món đó.

In [ ]:
# For negatives: check if retriever returns HIGH confidence results
fp_rows = []
for mname, ret in methods.items():
    fp_count = 0
    for c in negative:
        results = ret.search(c["query"], top_k=5)
        if results and results[0].score > 0:
            fp_count += 1
    fp_rows.append({
        "Phương pháp": mname,
        "Trả về kết quả": f"{fp_count}/{len(negative)}",
        "Tỷ lệ false positive": f"{fp_count/len(negative):.0%}",
    })
display(pd.DataFrame(fp_rows).style.hide(axis="index").set_caption(
    "False positive trên negative cases"))

# Show example false positive
print("\nVí dụ negative case và kết quả retrieval:")
for c in negative[:3]:
    print(f"\n  Query: {c['query']}")
    for mname, ret in methods.items():
        results = ret.search(c["query"], top_k=1)
        if results:
            print(f"    {mname:12s}: {results[0].score:.4f}  {results[0].chunk.source} [{results[0].chunk.title}]")
        else:
            print(f"    {mname:12s}: (no results)")

#### Nhận xét False Positive

Tất cả retriever đều trả về kết quả cho negative cases — đây là hạn chế
của retrieval nói chung: chúng luôn trả về gần nhất, không biết nói "không có".

**Giải pháp trong production:** Guardrail flags + LLM để phán đoán
"document này có thực sự trả lời câu hỏi không?" trước khi respond.

### 6.3 Phân tích case — ai cứu ai?

Trên 107 KB cases, phân tích BM25 vs Dense:

In [ ]:
def check_hit(retriever, case):
    if not case.get("expected_selectors"): return False
    results = retriever.search(case["query"], top_k=5)
    sources = {r.chunk.source for r in results}
    return any(any(sel.split(":")[-1] in s for s in sources) for sel in case["expected_selectors"])

dense_saves, bm25_saves, both_miss, both_hit = [], [], [], []
for c in kb_cases:
    b, d = check_hit(bm25, c), check_hit(dense, c)
    if b and d: both_hit.append(c)
    elif d and not b: dense_saves.append(c)
    elif b and not d: bm25_saves.append(c)
    else: both_miss.append(c)

print(f"Cả hai hit:       {len(both_hit)}")
print(f"Dense cứu BM25:   {len(dense_saves)}")
print(f"BM25 cứu Dense:   {len(bm25_saves)}")
print(f"Cả hai miss:      {len(both_miss)}")

venn = pd.DataFrame([{"": "Dense HIT", "BM25 HIT": len(both_hit), "BM25 MISS": len(dense_saves)},
                      {"": "Dense MISS", "BM25 HIT": len(bm25_saves), "BM25 MISS": len(both_miss)}])
display(venn.style.hide(axis="index").set_caption("Ma trận BM25 vs Dense (107 KB cases)"))

#### Dense cứu được mà BM25 miss

In [ ]:
if dense_saves:
    ds = [{"Query": c["query"][:50], "Noise": c.get("noise_type","clean"),
           "Expected": c["expected_selectors"][0].split(":")[-1]} for c in dense_saves[:8]]
    display(pd.DataFrame(ds).style.hide(axis="index").set_caption(f"Dense cứu BM25 ({len(dense_saves)} cases)"))
else: print("Không có")

#### BM25 cứu được mà Dense miss

In [ ]:
if bm25_saves:
    bs = [{"Query": c["query"][:50], "Noise": c.get("noise_type","clean"),
           "Expected": c["expected_selectors"][0].split(":")[-1]} for c in bm25_saves[:8]]
    display(pd.DataFrame(bs).style.hide(axis="index").set_caption(f"BM25 cứu Dense ({len(bm25_saves)} cases)"))
    noise_types = Counter(c.get("noise_type","clean") for c in bm25_saves)
    print(f"\nPhân loại: {dict(noise_types)}")
    print("=> BM25 cứu Dense chủ yếu là noisy cases (nhờ normalize built-in)")
else: print("Không có")

### 6.4 Error analysis — các case khó

Phân tích các cases mà **cả 3 methods đều miss** — tại sao?

In [ ]:
hybrid_saves = [c for c in both_miss if check_hit(hybrid, c)]
still_miss = [c for c in both_miss if not check_hit(hybrid, c)]

print(f"Cả BM25+Dense miss: {len(both_miss)}")
print(f"Hybrid cứu:         {len(hybrid_saves)}")
print(f"Cả 3 miss:          {len(still_miss)}")

# Deduplicate: group by base query
base_queries = {}
for c in still_miss:
    base = c.get("parent_case_id", c["case_id"])
    base_queries.setdefault(base, []).append(c)
print(f"\nThực tế là {len(base_queries)} queries gốc + noisy variants của chúng:")

for base_id, cases in base_queries.items():
    c0 = cases[0]
    expected_file = c0["expected_selectors"][0].split(":")[-1]
    print(f"\n  Query gốc: {[x for x in cases if x.get('noise_type') in (None, 'clean', 'vocab-mismatch')][0]['query'] if any(x.get('noise_type') in (None, 'clean', 'vocab-mismatch') for x in cases) else cases[0]['query']}")
    print(f"  Expected:   {expected_file}")
    print(f"  Variants:   {len(cases)} cases ({', '.join(c.get('noise_type','clean') for c in cases)})")
    # Show what BM25 returned for first case
    results = bm25.search(cases[0]["query"], top_k=3)
    top = [f"{r.chunk.source}" for r in results[:3]]
    print(f"  BM25 top-3: {", ".join(top)}")

#### Tại sao cả 3 miss?

8 cases miss thực chất là **vài queries gốc** + các noisy variants của chúng.
Nguyên nhân chính:

1. **Từ đồng nghĩa tiếng Việt:** "đậu phộng" và "lạc" là cùng một thứ nhưng
   BM25 không match, và E5 (train đa ngôn ngữ) cũng không nắm được.
2. **Noisy + semantic gap:** Không dấu ("dau phong") làm mất cả lexical lẫn semantic signal.
3. **Vocab mismatch sâu:** "hủy đơn" vs "Thay Đổi Order" — khác hoàn toàn về từ.

Đây là giới hạn của retrieval — cần LLM hoặc knowledge graph để hiểu
"đậu phộng = lạc" hoặc "hủy đơn ≈ thay đổi order".

### 6.5 Thí nghiệm: Normalize giúp Dense không?

Dense E5 chỉ 25% trên noisy. Hệ thống có 2 hàm normalize:

| Hàm | Thao tác | Phù hợp |
|---|---|---|
| `normalize_query_text()` | **Strip dấu**, sửa teencode, emoji | BM25 (lexical) |
| `normalize_vietnamese()` | **Giữ dấu**, chỉ sửa teencode | Dense (semantic) |

**Tại sao không dùng `normalize_query_text()` cho Dense?**
Vì nó strip dấu → query thành text không dấu → E5 (train trên text có dấu) hiểu sai.
Nhưng `normalize_vietnamese()` cũng có hạn chế — xem kết quả:

In [ ]:
from app.rag.vietnamese_normalizer import normalize_vietnamese
from app.rag.retriever import RetrievedChunk

class NormalizedDense:
    def __init__(self, dense_ret):
        self._d = dense_ret
    def search(self, query, top_k=5, **kw):
        return self._d.search(normalize_vietnamese(query), top_k=top_k, **kw)

dense_norm = NormalizedDense(dense)
hybrid_norm = HybridRrfRetriever([bm25, dense_norm])

# Full comparison
configs = {"BM25": bm25, "Dense": dense, "Dense+norm_vi": dense_norm,
           "Hybrid": hybrid, "Hybrid+norm_vi": hybrid_norm}

rows = []
for gname, gcases in [("Clean KB", clean_kb), ("Noisy", noisy), ("Vocab", vocab)]:
    n = len(gcases)
    row = {"Tập": f"{gname} ({n})"}
    for cname, cret in configs.items():
        h1, h5 = eval_hits(cret, gcases)
        row[cname] = f"{h5}/{n} ({h5/n:.0%})"
    rows.append(row)
display(pd.DataFrame(rows).style.hide(axis="index").set_caption(
    "Impact của normalize_vietnamese() lên Dense"))

#### Tại sao Dense+norm_vi giảm trên Clean?

`normalize_vietnamese()` thay đổi một số từ trong query **cũng như dấu câu** →
embedding thay đổi → cosine similarity giảm.

**Quan trọng:** Documents được encode **không normalize**.
Nếu normalize query nhưng không normalize document → **embedding space mismatch**.

**Kết luận:** Normalize cho Dense chỉ nên áp dụng **có điều kiện** (khi detect query là noisy),
không áp dụng mặc định. Hoặc cần normalize **cả documents lẫn queries**.

### 6.6 Impact Normalize và Variants trên BM25

In [ ]:
import math
from app.rag.knowledge_base import KnowledgeChunk
import re as _re

class RawBM25:
    def __init__(self, chunks):
        self._chunks = chunks
        self._tok = [c.content.lower().split() for c in chunks]
        self._N = len(chunks)
        self._avgdl = sum(len(t) for t in self._tok) / max(self._N, 1)
        self._df = {}
        for tokens in self._tok:
            for t in set(tokens): self._df[t] = self._df.get(t, 0) + 1
    def search(self, query, top_k=5, **kw):
        qtok = query.lower().split()
        scores = []
        for chunk, dtok in zip(self._chunks, self._tok):
            s = 0.0
            dl = len(dtok)
            for qt in qtok:
                tf = dtok.count(qt)
                df = self._df.get(qt, 0)
                if tf == 0 or df == 0: continue
                idf = math.log((self._N - df + 0.5)/(df + 0.5) + 1)
                s += idf * (tf*2.0)/(tf + 1.2*(1-0.75+0.75*dl/self._avgdl))
            if s > 0: scores.append(RetrievedChunk(chunk=chunk, score=s))
        scores.sort(key=lambda x: x.score, reverse=True)
        return scores[:top_k]

bm25_raw = RawBM25(kb_chunks)
stripped = [KnowledgeChunk(source=c.source, title=c.title,
    content=_re.sub(r"question_variants:.*", "", c.content),
    tags=c.tags, chunk_id=c.chunk_id, risk_tier=c.risk_tier) for c in kb_chunks]
bm25_no_var = BM25Retriever(stripped)

bm25_cfgs = {"BM25 production": bm25, "BM25 (no normalize)": bm25_raw, "BM25 (no variants)": bm25_no_var}
rows = []
for gname, gcases in [("Clean KB", clean_kb), ("Noisy", noisy), ("Vocab", vocab)]:
    n = len(gcases)
    row = {"Tập": f"{gname} ({n})"}
    for cname, cret in bm25_cfgs.items():
        h1, h5 = eval_hits(cret, gcases)
        row[cname] = f"{h5}/{n} ({h5/n:.0%})"
    rows.append(row)
display(pd.DataFrame(rows).style.hide(axis="index").set_caption(
    "Impact Normalize và Variants trên BM25"))

### 6.7 Latency

In [ ]:
test_qs = ["nhà hàng có wifi không?", "món nào không cay?",
           "thanh toán bằng thẻ?", "có combo cho 4 người?", "hủy món như nào?"]

latency = {}
for name, ret in methods.items():
    times = []
    for q in test_qs:
        t0 = time.perf_counter()
        ret.search(q, top_k=5)
        times.append((time.perf_counter()-t0)*1000)
    latency[name] = times

lat_df = pd.DataFrame([{"Phương pháp": n, "Min": f"{min(t):.1f}ms",
    "Avg": f"{sum(t)/len(t):.1f}ms", "Max": f"{max(t):.1f}ms"}
    for n, t in latency.items()])
display(lat_df.style.hide(axis="index").set_caption("Latency per query"))

fig, ax = plt.subplots(figsize=(6, 4))
avg = [sum(t)/len(t) for t in latency.values()]
bars = ax.bar(latency.keys(), avg, color=["#3b82f6","#8b5cf6","#10b981"])
for b, v in zip(bars, avg):
    ax.text(b.get_x()+b.get_width()/2., b.get_height()+0.3,
            f"{v:.1f}ms", ha="center", fontsize=10)
ax.set_ylabel("Avg latency (ms)")
ax.set_title("Latency trung bình per query")
plt.tight_layout()
plt.show()

### 6.8 Tổng hợp — Heatmap

In [ ]:
import matplotlib.colors as mcolors

# Build heatmap data
hm_methods = ["BM25", "Dense E5", "Hybrid RRF"]
hm_groups = ["Clean KB", "Noisy", "Vocab mismatch"]
hm_data = []
for g in hm_groups:
    row = []
    gcases = {"Clean KB": clean_kb, "Noisy": noisy, "Vocab mismatch": vocab}[g]
    for m in hm_methods:
        _, h5 = eval_hits(methods[m], gcases)
        row.append(h5 / len(gcases))
    hm_data.append(row)

fig, ax = plt.subplots(figsize=(7, 4))
cmap = mcolors.LinearSegmentedColormap.from_list("", ["#fee2e2","#fef9c3","#dcfce7"])
im = ax.imshow(hm_data, cmap=cmap, vmin=0.2, vmax=1.0, aspect="auto")

ax.set_xticks(range(len(hm_methods)))
ax.set_xticklabels(hm_methods)
ax.set_yticks(range(len(hm_groups)))
ax.set_yticklabels(hm_groups)

for i in range(len(hm_groups)):
    for j in range(len(hm_methods)):
        ax.text(j, i, f"{hm_data[i][j]:.0%}", ha="center", va="center",
                fontsize=14, fontweight="bold")

ax.set_title("Hit@5 Heatmap — 3 methods \u00d7 3 tập dữ liệu")
plt.colorbar(im, ax=ax, label="Hit@5")
plt.tight_layout()
plt.show()

### 6.9 Export screening metrics (Part II → JSON)

In [ ]:
import json
from datetime import datetime, timezone

h1, h5 = eval_hits(hybrid, kb_cases)
hit5_overall = h5 / len(kb_cases) if kb_cases else 0.0
by_group = {}
for gname, gcases in [("clean_kb", clean_kb), ("noisy", noisy), ("vocab", vocab)]:
    _, g5 = eval_hits(hybrid, gcases)
    by_group[gname] = {"hit5": g5 / len(gcases) if gcases else 0.0, "n": len(gcases)}
screening = {
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S"),
    "method": "hybrid_rrf",
    "hit5_overall": hit5_overall,
    "hit5_by_group": by_group,
    "eval_case_count": len(kb_cases),
}
screen_path = AI_ROOT / "evaluation" / "results" / "notebook_retrieval_screening.json"
screen_path.parent.mkdir(parents=True, exist_ok=True)
screen_path.write_text(json.dumps(screening, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Hybrid Hit@5 overall: {hit5_overall:.1%} ({h5}/{len(kb_cases)} cases)")
print(f"Saved {screen_path}")

### Hit@5: screening notebook vs release gate

| Bộ | N | Artifact | Mục đích |
|---|---:|---|---|
| **Screening notebook** | 107 | `notebook_retrieval_screening.json` | So sánh BM25 / Dense / Hybrid trong Part II (execute Part II) |
| **Release dev gate** | 110 | `dev_retrieval_summary.v3.json` | Gate staging (`AI_STAGING_READINESS.md`); khác bộ case và có thể khác cấu hình E5 |

> Hai con số **không** so sánh trực tiếp — cột Hybrid trong §15 là **screening**, không phải release 99%.

---
## Kết luận Phần II

### Lưu ý về dữ liệu và thống kê

- Toàn bộ kết quả trên **bộ dữ liệu tự xây** (26 files KB, 107 eval cases).
- Với n nhỏ (vocab=19, clean=35), khác biệt nhỏ có thể do ngẫu nhiên.
- Kết quả không generalize — nhưng phương pháp đánh giá có thể tái sử dụng.

### Những điều đã chứng minh

1. **Không có silver bullet** — mỗi method có thế mạnh riêng:
   - BM25: mạnh nhất noisy (89%) nhờ normalize
   - Dense: mạnh nhất clean (100%) và vocab (89%) nhờ semantic
   - Hybrid: cân bằng nhất

2. **Normalize phải phù hợp loại retriever:**
   - BM25: `normalize_query_text()` (strip dấu) → +49% noisy
   - Dense: `normalize_vietnamese()` (giữ dấu) → cải thiện noisy nhưng giảm clean
   - Dense normalize chỉ nên áp dụng khi detect query noisy

3. **False positive là vấn đề chung** — cả 3 methods đều trả về kết quả
   cho negative cases. Cần guardrails ở tầng LLM.

### Trade-off

| Tiêu chí | BM25 | Dense E5 | Hybrid RRF |
|---|---|---|---|
| **Clean** | 91% | **100%** | **100%** |
| **Noisy** | **89%** | 25% | 74% |
| **Vocab** | 79% | **89%** | **89%** |
| **Latency** | ~3ms | ~17ms | ~26ms |
| **GPU** | Không | Không (CPU đủ) | Không |

### Khuyến nghị

- **Production:** Hybrid RRF (BM25 + normalize + Dense E5, encoder **`e5_small`**)
- **Fallback:** BM25 + normalize + variants (88% tổng, 3ms latency)
- **Cải thiện tiếp:** Thêm eval cases từ traffic thật, thử fine-tune E5

> **Kết luận:** Trên bộ dữ liệu tự xây, kết hợp là chiến lược tốt nhất.
> Mỗi phương pháp bù nhược điểm cho nhau — BM25 cứu noisy, Dense cứu semantic.

> **Chốt báo cáo:** cuối notebook — **§18 Đưa vào production** (tính năng đã áp dụng + stack vận hành).

# PHẦN III — CHATBOT CÓ NGỮ CẢNH

Phần I đã khám phá **26 files KB, 213 chunks** và cách normalize tiếng Việt.
Phần II đã chứng minh **Hybrid RRF** là retrieval tốt nhất (Hit@5 trong `notebook_retrieval_screening.json` sau khi chạy Part II).

Giờ câu hỏi là: khi khách hỏi `"có wifi không?"` vs `"gợi ý món đi"`,
hệ thống quyết định dùng **Hybrid RRF** (tìm KB) hay **Catalog API** (tìm menu) như thế nào?

> **Thống nhất dữ liệu:** Part III–IV dùng cùng **20 queries** (6 categories)
> cho cả intent classification, pipeline test, và so sánh 3 model.
> Đảm bảo kết quả traceable: query X → intent Y → route Z → response.

Phần này trình bày 4 thành phần xây dựng pipeline hoàn chỉnh:

```
User query
  │
  ├── [1] Guardrails      → chặn PII, injection, off-topic
  ├── [2] Intent Router    → KB (Hybrid RRF) / Catalog / Cart / Fallback
  ├── [3] Session Memory   → nhớ dị ứng, lịch sử, tóm tắt
  └── [4] Claim Verifier   → chặn hallucination sau LLM
```

In [ ]:
import json
from scripts.notebook_metrics import load_retrieval_headlines

live_test = json.load(open(
    AI_ROOT / "evaluation" / "results" / "notebook_live_test.json",
    encoding="utf-8"
))
_retrieval_headlines = load_retrieval_headlines(AI_ROOT)
hit5_label = _retrieval_headlines["screening_label"]
hit5_score = float(_retrieval_headlines["screening_hit5"] or 0.0)
release_retrieval_note = _retrieval_headlines.get("release_label") or ""
print(f"Test: {live_test['timestamp']}, Model: {live_test['model']}, Retrieval: {live_test['retrieval_method']}")
print(f"Part II screening: {hit5_label}")
if release_retrieval_note:
    print(f"Release artifact: {release_retrieval_note}")

## 8. Evidence Routing — quyết định đường đi của query

Phần II chứng minh **Hybrid RRF** là retrieval tốt nhất.
Nhưng không phải mọi query đều cần search KB —
`"gợi ý món"` cần menu data, không cần KB.

**Intent Classifier** làm router: phân loại query → chọn đường đi.

| Intent | Route | Nguồn dữ liệu |
|---|---|---|
| `restaurant_info`, `payment` | **KB (Hybrid RRF)** | 213 chunks từ Part I |
| `browse_menu`, `ask_price` | **Catalog API** | Menu data thật |
| `order` | **Cart API** | Giỏ hàng |
| `general` | **Fallback** | Hỏi lại / từ chối |

> **Nguồn dữ liệu:** `notebook_live_test.json` → `intent_results` (20 queries, cùng set với §12 và §14)

In [ ]:
# Bảng intent classification
ir = live_test["intent_results"]
route_map = {"browse_menu": "Catalog API", "ask_price": "Catalog API",
             "order": "Cart API", "restaurant_info": "KB (Hybrid RRF)",
             "payment": "KB (Hybrid RRF)", "spice_level": "KB (Hybrid RRF)",
             "recommend": "Catalog API", "allergy": "KB + Menu",
             "general": "Fallback"}
intent_rows = [{"Query": r["query"][:35], "Intent": r["intent"],
                "Conf": f'{r["confidence"]:.2f}',
                "Route": route_map.get(r["intent"], "Fallback")}
               for r in ir]
display(pd.DataFrame(intent_rows).style.hide(axis="index").set_caption(
    f"Intent Classification: {len(ir)} queries"))

In [ ]:
# Routing Distribution
from collections import Counter
import matplotlib
matplotlib.rcParams["figure.dpi"] = 150
routes = [route_map.get(r["intent"], "Fallback") for r in ir]
rc = Counter(routes)

fig, ax = plt.subplots(figsize=(6, 4))
labels = list(rc.keys())
sizes = list(rc.values())
colors = ["#10b981", "#3b82f6", "#f59e0b", "#94a3b8", "#8b5cf6"][:len(labels)]
wedges, texts, autotexts = ax.pie(sizes, labels=labels, colors=colors,
    autopct="%1.0f%%", startangle=90, pctdistance=0.75)
for t in texts: t.set_fontsize(9)
for t in autotexts: t.set_fontsize(8)
ax.set_title("Routing Distribution", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Intent Confidence
classified = [r for r in ir if r["intent"] != "general"]
fallback = [r for r in ir if r["intent"] == "general"]
intents_sorted = sorted(classified, key=lambda x: x["confidence"])
names = [r["intent"] for r in intents_sorted]
confs = [r["confidence"] for r in intents_sorted]

fig, ax = plt.subplots(figsize=(7, max(3, len(names)*0.4)))
bars = ax.barh(range(len(names)), confs, color="#3b82f6", height=0.5)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8)
for i, v in enumerate(confs):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=8)
ax.set_xlabel("Confidence", fontsize=9)
ax.set_title(f"Intent Confidence ({len(classified)} classified, {len(fallback)} fallback)",
             fontsize=11, fontweight="bold")
ax.set_xlim(0, max(confs)*1.3 if confs else 1)
plt.tight_layout()
plt.show()

### Nhận xét Evidence Routing

**Kết quả tốt:**
- Intent classifier phân biệt được các loại query chính: menu, giá, thông tin nhà hàng, thanh toán
- Routing đúng: FAQ → KB (Hybrid RRF từ Part II), menu → Catalog API
- Tốc độ <1ms (rule-based, không ML)

**Hạn chế:**
- **Teencode** không match: `"ko cay dc ko"` → `general` (thực ra là `spice_level`)
- **Confidence thấp** cho nhiều intents (0.1-0.3) — vì chỉ đếm keyword, không scoring phức tạp
- **`"tôi dị ứng tôm"` → `browse_menu`** thay vì `allergy` — cần thêm allergy rules

**Liên kết Part II:**
Khi intent là `restaurant_info`, `payment`, `spice_level` →
hệ thống gọi **Hybrid RRF** (BM25 + Dense E5) đã benchmark ở Part II
với Hit@5 screening từ Part II (`notebook_retrieval_screening.json`). Đây là lúc retrieval được sử dụng thật.

## 9. Guardrails — bảo vệ chatbot

**Trước khi** query vào pipeline, regex patterns kiểm tra các mối nguy:
- **PII:** Số CCCD, số điện thoại → không được lưu/xử lý
- **Injection:** Cố thay đổi hành vi chatbot
- **Off-topic:** Câu hỏi không liên quan nhà hàng
- **Profanity:** Ngôn ngữ thô tục
- **Fabrication:** Yêu cầu AI bịa giá/món

> **Nguồn dữ liệu:** `notebook_live_test.json` → `guard_results` (10 scenarios thử cố định)

In [ ]:
gr = live_test["guard_results"]
guard_rows = [{"Query": r["query"][:42], "Scenario": r["scenario"],
               "Flags": ", ".join(r["flags"])}
              for r in gr]
display(pd.DataFrame(guard_rows).style.hide(axis="index").set_caption(
    f"Guardrails: {len(gr)} test cases"))

In [ ]:
# Chart 2: Flag distribution
all_flags = []
for r in gr:
    if r["flags"] != ["CLEAN"]:
        all_flags.extend(r["flags"])
fc = Counter(all_flags)

fig, ax = plt.subplots(figsize=(10, 4))
flags_sorted = sorted(fc.items(), key=lambda x: x[1], reverse=True)
names = [f[0].replace("_"," ").title() for f in flags_sorted]
vals = [f[1] for f in flags_sorted]
colors_f = ["#ef4444","#f59e0b","#8b5cf6","#3b82f6","#10b981","#ec4899","#6366f1"]
bars = ax.barh(names, vals, color=colors_f[:len(names)])
for b, v in zip(bars, vals):
    ax.text(b.get_width()+0.1, b.get_y()+b.get_height()/2, str(v), va="center")
ax.set_xlabel("Số lần")
ax.set_title("Phân bố Guardrail Flags")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Nhận xét Guardrails

**Điểm mạnh:**
- **7 loại flags** phát hiện được, bao phủ các mối nguy chính
- Phân biệt đúng: `"đặt luôn 2 tô phở"` → CONFIRMATION (cần xác nhận),
  nhưng `"thanh toán bằng thẻ"` → Clean (chỉ hỏi thông tin)
- Prompt injection, PII → chặn ngay, không vào pipeline

**Hạn chế:**
- Regex không bắt paraphrase: `"làm ơn quên hết đi"` có thể bypass injection
- Không detect tone tiêng (mỉa mai) — cần NLU

**Trong pipeline:** Guardrails chạy **trước** Intent Router và **trước** Hybrid RRF.
Nếu bị flag → không tốn tài nguyên gọi LLM.

## 10. Session Memory — nhớ ngữ cảnh qua nhiều lượt

Khi khách hỏi nhiều lượt, hệ thống phải **nhớ**:
- Lượt 1: `"tôi dị ứng tôm"` → lưu constraint
- Lượt 2: `"gợi ý món"` → lọc bỏ món có tôm
- Lượt 3: `"cái đó bao nhiêu?"` → hiểu `"cái đó"` = món vừa gợi

Hệ thống dùng **3 cơ chế**:

In [ ]:
from app.schemas import SessionState, LiveContext

# 3 cơ chế giữ ngữ cảnh
mechanisms = [
    {"Cơ chế": "Chat History", "Mô tả": "Các lượt trước gửi vào LLM prompt",
     "Ví dụ": "LLM thấy lượt 1 hỏi phở bò → hiểu 'cái đó'"},
    {"Cơ chế": "Rolling Summary", "Mô tả": "LLM tóm tắt hội thoại mỗi lượt",
     "Ví dụ": "Khách dị ứng tôm, đã gợi ý phở bò..."},
    {"Cơ chế": "Typed Session State", "Mô tả": "Pydantic model lưu constraints",
     "Ví dụ": "allergens=[\"tôm\"], budget=200000"},
]
display(pd.DataFrame(mechanisms).style.hide(axis="index").set_caption(
    "3 cơ chế giữ ngữ cảnh"))

# SessionState fields
ss_rows = [{"Field": name, "Type": str(field.annotation).replace("typing.","")[:25]}
           for name, field in SessionState.model_fields.items()]
display(pd.DataFrame(ss_rows).style.hide(axis="index").set_caption(
    "SessionState — typed constraints"))

### 10.1 Dữ liệu thật từ nhà hàng (LiveContext)

AI **không bịa** giá, không bịa món — trả lời dựa trên **dữ liệu thật** từ database.
Mỗi request, frontend gửi kèm `LiveContext` chứa menu, giỏ hàng, khuyến mãi hiện tại:

In [ ]:
# LiveContext fields
lc_descs = {"catalog_version": "Phiên bản menu", "menu_items": "Món: tên, GIÁ THẬT, allergens, is_available",
            "cart_items": "Giỏ hàng hiện tại", "orders": "Đơn đã đặt",
            "promotions": "Khuyến mãi đang chạy", "local_time": "Giờ hiện tại",
            "meal_period": "Buổi ăn (lunch/dinner)", "table_code": "Mã bàn"}
lc_rows = [{"Field": name, "Type": str(field.annotation).replace("typing.","")[:25],
            "Mô tả": lc_descs.get(name, "")}
           for name, field in LiveContext.model_fields.items()]
display(pd.DataFrame(lc_rows).style.hide(axis="index").set_caption(
    "LiveContext — dữ liệu thật mỗi request"))

### 10.2 Cart Suggestion — gợi ý thêm vào giỏ

Khi AI tư vấn món, response chứa `suggested_cart_actions`:
```json
{"content": "Phở Bò Tái Nạm (75.000đ) là best-seller...",
 "suggested_cart_actions": [{"menu_item_id": "pho-bo-01", "name": "Phở Bò Tái Nạm"}]}
```
Frontend nhận `menu_item_id` → hiện **nút "Đặt món"** — khách nhấn 1 lần là thêm giỏ.
Không cần gõ lại tên món.

### 10.3 Session Persistence

| Tính năng | Cơ chế |
|---|---|
| **Refresh trang không mất hội thoại** | `session_id` lưu trên server, load lại khi reconnect |
| **Hội thoại gắn với phiên bàn** | `table_code` → đóng bàn mới reset chat |
| **Nhớ dị ứng suốt phiên** | `session_state.constraints.allergens` persist |
| **Không gợi lại món bị từ chối** | `rejected_menu_item_ids` trong SessionState |

Frontend gửi `session_id` + `table_code` mỗi request.
Khách đổi điện thoại, refresh → vẫn thấy lịch sử chat.
Đóng bàn (thanh toán xong) → session bị hủy, bàn mới = chat mới.

### Nhận xét Session Memory

**3 cơ chế bổ sung cho nhau:**
- **Chat History** giúp LLM hiểu context gần ("cái đó" = món vừa nói)
- **Rolling Summary** giữ thông tin xa (dị ứng từ lượt đầu)
- **Typed SessionState** đảm bảo constraints (allergens, budget) không bị LLM quên

**LiveContext là chốt an toàn:** AI không thể bịa giá vì mọi response
đều được cross-check với `menu_items` thật từ database.
Nếu `is_available=False`, AI tự động không gợi ý món đó.

**Cart Suggestion** biến chatbot từ "hỏi-đáp" thành **conversion tool** —
khách không cần gõ tên món, nhấn nút là thêm giỏ.

> **Liên kết Part II:** Khi intent là `restaurant_info`, Session Memory
> gửi query tới **Hybrid RRF** (kết quả screening Part II, xem `hit5_label` ở đầu Part III).
> Khi intent là `browse_menu`, dùng LiveContext thay vì KB.

## 11. Claim Verifier — chống hallucination

LLM có thể bịa số liệu. Claim Verifier kiểm tra **sau** khi LLM trả lời:
- Claim về chính sách/FAQ có khớp **evidence từ KB (Part I)**?
- Claim về món, giá, category, tag có khớp **LiveContext menu**?
- Evidence ID có tồn tại trong tập evidence được phép của đúng request?
- Nếu không → chặn response, trả về `"chưa đủ thông tin"`

> **Nguồn dữ liệu:** `notebook_live_test.json` → `claim_results` (4 claims, kiểm tra KB Part I)

In [ ]:
cr = live_test["claim_results"]
claim_rows = [{"Claim": r["text"][:45],
               "Evidence": (r["evidence_ids"][0][:20]+"..." if r["evidence_ids"] else "(none)"),
               "Và lý": "✅ OK" if r["verified"] else "❌ Fail",
               "Lý do": r["reason"] or "Khớp evidence"}
              for r in cr]
display(pd.DataFrame(claim_rows).style.hide(axis="index").set_caption(
    f"Claim Verification: {len(cr)} claims"))

In [ ]:
# Claim Verification
ok_c = len([r for r in cr if r["verified"]])
fail_c = len(cr) - ok_c
reasons = Counter(r["reason"] for r in cr if r["reason"])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].bar(["Verified", "Rejected"], [ok_c, fail_c],
            color=["#10b981", "#ef4444"], width=0.5)
axes[0].set_title("Claim Verification", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate([ok_c, fail_c]):
    axes[0].text(i, v + 0.05, str(v), ha="center", fontsize=10, fontweight="bold")

if reasons:
    r_names = [k.replace("_"," ").title()[:25] for k in reasons.keys()]
    r_vals = list(reasons.values())
    axes[1].barh(r_names, r_vals, color="#ef4444", height=0.4)
    axes[1].set_title("Rejection Reasons", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("Count")
else:
    axes[1].text(0.5, 0.5, "No rejections", ha="center", va="center",
                transform=axes[1].transAxes, fontsize=12)
    axes[1].set_title("Rejection Reasons", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

### Nhận xét Claim Verifier

Verifier bắt được **3 loại hallucination**:

| Loại | Ví dụ | Cơ chế |
|---|---|---|
| **Số sai** | 8:00 thay vì 10:00 | So sánh numeric với evidence |
| **Không evidence** | "có hồ bơi" nhưng không KB nào nói | Check `evidence_ids` rỗng |
| **Evidence giả** | ID không tồn tại | Lookup trong KB chunks + live menu IDs |

**Nguồn sự thật:** Verifier dùng 213 KB chunks cho FAQ/chính sách và LiveContext cho món/giá/category/tag. Evidence assembly này dùng chung cho cả ba profile.

**Hạn chế:** Lexical matching — nếu LLM paraphrase đúng nghĩa nhưng khác từ,
verifier có thể false-reject.

## 11.1 Ba phương pháp pipeline được nghiên cứu

Notebook **không mặc định** `planner + typed state` là kiến trúc production. Ba profile dưới đây phải chạy trên cùng DeepSeek, menu, KB, prompt budget và dataset:

| Profile | Cách xử lý factual | Câu phức tạp | Bộ nhớ hội thoại |
|---|---|---|---|
| `llm_first_v1` | DeepSeek-first (trừ guardrail/fast path bắt buộc) | DeepSeek | Rolling summary |
| `evidence_first_v2` | Menu/giá/category/tag deterministic từ live evidence | DeepSeek | Rolling summary |
| `planner_state_v3` | Evidence-first | DeepSeek semantic planner rồi DeepSeek trả lời khi cần | Typed `ConversationFrame` + constraints |

Mọi profile dùng chung evidence assembly và Claim Verifier. Vì vậy kết quả so sánh đo khác biệt phương pháp, không bị thiên vị bởi lỗi evidence của riêng một nhánh.

---
# PHẦN IV — THỰC NGHIỆM

Phần III giải thích từng thành phần riêng.
Phần này gửi **cùng 20 queries** qua **toàn bộ pipeline kết hợp**:

```
20 queries (cùng set với §8 Intent)
  → Guardrails (§9) → Intent Router (§8)
  → Hybrid RRF (Part II) hoặc Catalog API
  → LLM (DeepSeek) + Session (§10)
  → Claim Verifier (§11) → Response
```

> Cùng query `"có wifi không?"` đã được intent classify ở §8
> → giờ chạy qua full pipeline để trace kết quả.

## Tài liệu tái lạp (reproducibility)

Mọi số trong Part IV–V lấy từ JSON trong `evaluation/results/`.
Chạy lại artifact **cùng phiên** (9router bật) trước khi so sánh §12 và §14.

| Lệnh | Output |
|---|---|
| `py scripts/_run_live_tests.py` | `notebook_live_test.json` (§12–§13) |
| `py scripts/_dual_model_test.py` | `dual_model_test.json` (§14) |
| `py scripts/build_research_notebook.py` | Build + validate cells |
| `py scripts/build_research_notebook.py --execute` | Execute notebook (Part II → screening JSON) |
| `py scripts/build_research_notebook.py --regen-live --execute` | Regen live JSON rồi execute |

> Thứ tự khuyến nghị: `--execute` → `--regen-live` → `--execute` lần 2.
> Env: `LLM_PROVIDER=9router`, model production theo `docs/ai/AI_STAGING_READINESS.md`.

In [ ]:
from IPython.display import Markdown, display
from scripts.notebook_metrics import format_artifact_provenance_table

display(Markdown(format_artifact_provenance_table(AI_ROOT)))

In [ ]:
import subprocess
from scripts.notebook_metrics import summarize_live_test

def _git_head():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"],
            cwd=PROJECT_ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError, OSError):
        return "unknown"

live_snap = summarize_live_test(live_test)
print(f"Git (repo root): {_git_head()}")
print(f"notebook_live_test.json: {live_snap['timestamp']}  model={live_snap['model']}")
dual_path = AI_ROOT / "evaluation" / "results" / "dual_model_test.json"
if dual_path.exists():
    _dual_ts = json.loads(dual_path.read_text(encoding="utf-8")).get("timestamp", "?")
    print(f"dual_model_test.json: {_dual_ts}")
else:
    print("dual_model_test.json: (chưa có — chạy _dual_model_test.py)")

## Bảng thuật ngữ metric (Part IV)

| Metric | § | Ý nghĪ | Denominator |
|---|---|---|---|
| **Detect rate** | III (§9–đ) | Guardrails / Claim Verifier bắt đúng case thiết kế | Số case trong bộ test riêng |
| **Pipeline availability** | §12 | Có response non-abstain trên `pipeline_results` | 20 query, **một** model (`notebook_live_test.json`) |
| **Non-abstain success** | §14 | `route != abstain` trên cùng 20 query, **3 model** | `dual_model_test.json` |
| **Strict success** | §12 (và glossary) | Abstain, fail-closed content, hoặc `route` unknown + flags block | `pipeline_results` / export fields `success_strict` |

> **Availability ≠ chất lượng câu trả lời.** Release gate: [`docs/ai/AI_STAGING_READINESS.md`](../../docs/ai/AI_STAGING_READINESS.md) (hiện **NOT READY**).

> Timestamp §12 và §14 có thể khác nhau nếu chạy artifact ở thời điểm khác nhau — đọc block repro phía trên trước khi so sánh.

## 12. Pipeline end-to-end: queries thật với LLM

> **Nguồn dữ liệu:** `notebook_live_test.json` → `pipeline_results`
> Cùng **20 queries** như §8 Intent. Đây là artifact pipeline lịch sử; thí nghiệm chọn kiến trúc mới ở §14.4 cố định model DeepSeek.

In [ ]:
pr = live_test["pipeline_results"]
pipe_rows = []
for r in pr:
    if "error" in r:
        pipe_rows.append({"Query": r["query"][:35], "Route": "ERROR",
                          "Nội dung": r.get("error","")[:40], "Latency": "?"})
    else:
        content = r.get("content","")[:70]
        if "<!-- question_variants" in content: content = "[KB fast-path]"
        pipe_rows.append({"Query": r["query"][:35], "Route": r["route"],
                          "Response": content[:50],
                          "Latency": f'{r["latency_ms"]:.0f}ms'})
display(pd.DataFrame(pipe_rows).style.hide(axis="index").set_caption(
    f"Pipeline end-to-end: {len(pr)} queries, model {live_test['model']}"))

In [ ]:
# Latency per Query
valid_pr = [r for r in pr if "error" not in r]
routes_p = [r["route"] for r in valid_pr]
latencies = [r["latency_ms"] for r in valid_pr]
qs = [r["query"][:22] for r in valid_pr]
rc_colors = {"kb_rag": "#10b981", "clarify": "#f59e0b", "abstain": "#ef4444",
             "live_data": "#3b82f6", "llm": "#8b5cf6"}
colors_l = [rc_colors.get(r, "#94a3b8") for r in routes_p]

fig, ax = plt.subplots(figsize=(8, max(4, len(qs)*0.3)))
bars = ax.barh(range(len(qs)), latencies, color=colors_l, height=0.6)
ax.set_yticks(range(len(qs)))
ax.set_yticklabels(qs, fontsize=7)
for i, (v, r) in enumerate(zip(latencies, routes_p)):
    ax.text(v + 50, i, f"{v:.0f}ms [{r}]", va="center", fontsize=7)
ax.set_xlabel("Latency (ms)", fontsize=9)
ax.set_title("Latency per Query", fontsize=11, fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Route Distribution
rc2 = Counter(routes_p)
fig, ax = plt.subplots(figsize=(6, 4))
labels = list(rc2.keys())
sizes = list(rc2.values())
colors_r = [rc_colors.get(r, "#94a3b8") for r in labels]
wedges, texts, autotexts = ax.pie(sizes, labels=labels, colors=colors_r,
    autopct="%1.0f%%", startangle=90, pctdistance=0.75)
for t in texts: t.set_fontsize(9)
for t in autotexts: t.set_fontsize(8)
ax.set_title("Route Distribution (Pipeline)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Phân tích pipeline theo route
from collections import Counter
route_stats = {}
for r in valid_pr:
    rt = r["route"]
    if rt not in route_stats: route_stats[rt] = {"count": 0, "lats": []}
    route_stats[rt]["count"] += 1
    route_stats[rt]["lats"].append(r["latency_ms"])
print("| Route | Số query | Avg Latency | Dùng Hybrid RRF? |")
print("|---|---|---|---|"  )
route_desc = {"kb_rag": "**Có** — Part II", "clarify": "Không — hỏi lại",
              "abstain": "Gọi LLM → Verifier chặn", "live_data": "Không — lookup"}
for rt, st in route_stats.items():
    avg = sum(st["lats"])/len(st["lats"])
    print(f"| `{rt}` | {st["count"]} | {avg:.0f}ms | {route_desc.get(rt, "?")} |")

In [ ]:
# Tóm tắt availability vs strict
from scripts.notebook_metrics import summarize_live_pipeline
pipe_sum = summarize_live_pipeline(live_test)
total_p = pipe_sum["pipeline_total"] or 1
print("### Nhận xét Pipeline (số liệu)")
print(f"Availability: {pipe_sum['availability_ok']}/{total_p} ({pipe_sum['availability_ok']/total_p:.0%})")
print(f"Strict: {pipe_sum['strict_ok']}/{total_p} ({pipe_sum['strict_ok']/total_p:.0%})")
print(f"route null/unknown: {pipe_sum['route_null']}, abstain: {pipe_sum['route_abstain']}")

In [ ]:
from IPython.display import Markdown, display
from scripts.notebook_metrics import format_part12_narrative

_dual_for_12 = None
if (AI_ROOT / "evaluation" / "results" / "dual_model_test.json").exists():
    _dual_for_12 = json.load(open(
        AI_ROOT / "evaluation" / "results" / "dual_model_test.json", encoding="utf-8"
    ))
display(Markdown(format_part12_narrative(live_test, _dual_for_12)))

> **Ghi chú kỹ thuật:** Claim Verifier + LiveContext vẫn là hướng cải thiện chính; nhận xét chi tiết theo từng query nằm ở bảng `pipeline_results` phía trên.

## 13. Multi-turn: giữ ngữ cảnh qua nhiều lượt

> **Nguồn dữ liệu:** `notebook_live_test.json` → `multi_results` (5 turns, model `cx/gpt-5.5`)

In [ ]:
mr = live_test["multi_results"]
for r in mr:
    if "error" in r:
        print(f"Turn {r['turn']}: ERROR")
        continue
    content = r.get("content","")[:80]
    if "<!-- question_variants" in content: content = "[KB fast-path]"
    print(f"Turn {r['turn']}: {r['query']}")
    print(f"  Route:  {r['route']}")
    print(f"  Answer: {content}")
    print()

In [ ]:
# Chart 6: Multi-turn success
ok_t = [r for r in mr if "error" not in r and r.get("route") != "abstain"]
has_sum = [r for r in mr if r.get("rolling_summary")]

fig, ax = plt.subplots(figsize=(6, 3))
metrics_mt = ["Trả lời OK", "Có summary"]
vals_mt = [len(ok_t)/len(mr), len(has_sum)/len(mr)]
bars = ax.bar(metrics_mt, vals_mt, color=["#10b981", "#3b82f6"])
for b, v in zip(bars, vals_mt):
    ax.text(b.get_x()+b.get_width()/2., b.get_height()+0.02,
            f"{v:.0%}", ha="center", fontweight="bold")
ax.set_ylim(0, 1.15)
ax.set_title(f"Multi-turn ({len(mr)} turns)")
plt.tight_layout()
plt.show()

### Nhận xét Multi-turn

5 turn thiết kế kiểm context. Bảng dưới lấy từ `multi_results` sau khi chạy thật.

In [ ]:
from IPython.display import Markdown, display
from scripts.notebook_metrics import format_part13_narrative

display(Markdown(format_part13_narrative(live_test)))

## 14. So sánh 3 model LLM (thí nghiệm lịch sử)

Test cùng **20 queries** (6 loại: KB FAQ, Menu, Allergy, Order, Off-topic, Chitchat)
với **3 model**:
- `oc/deepseek-v4-flash-free` — **triển khai** staging/production
- `cx/gpt-5.5` — quality gate / so sánh
- `cx/gpt-5.6-luna` — thế hệ mới (thí nghiệm)

Cùng pipeline, cùng KB (Part I), cùng Hybrid RRF (Part II), cùng Guardrails (§9).

> **Nguồn dữ liệu:** `dual_model_test.json` → cùng 20 queries như §8 và §12

In [ ]:
from scripts.notebook_metrics import summarize_dual_model, is_strict_pipeline_success

dual = json.load(open(
    AI_ROOT / "evaluation" / "results" / "dual_model_test.json",
    encoding="utf-8"
))
models = dual["models"]
total_q = len(dual["queries"])
_dual_summary = summarize_dual_model(dual)
print(f"Test: {dual['timestamp']}")
print(f"Models: {len(models)}")
print(f"Queries: {total_q}\n")
print(f"{'Model':35s} {'Avail':12s} {'Strict':12s}")
for m in models:
    stats = _dual_summary["per_model"][m]
    print(f"{m:35s} {stats['ok']}/{total_q} ({stats['ok']/total_q:.0%})   "
          f"{stats['strict_ok']}/{total_q} ({stats['strict_ok']/total_q:.0%})")

In [ ]:
# Bảng so sánh chi tiết
compare_rows = []
for i, qobj in enumerate(dual["queries"]):
    q = qobj["query"] if isinstance(qobj, dict) else qobj
    cat = qobj.get("category", "") if isinstance(qobj, dict) else ""
    row = {"Query": q[:25], "Cat": cat}
    for m in models:
        r = dual["results"][m][i]
        label = m.split("/")[1][:12]
        if "error" in r:
            row[f"{label}"] = "ERR"
        else:
            route = r["route"] or "llm"
            row[f"{label}"] = f'{route} {r["latency_ms"]:.0f}ms'
    compare_rows.append(row)
display(pd.DataFrame(compare_rows).style.hide(axis="index").set_caption(
    f"So sánh {len(models)} models x {total_q} queries"))

# Phân tích khác biệt giữa các model
diff_found = False
for i, qobj in enumerate(dual["queries"]):
    q = qobj["query"] if isinstance(qobj, dict) else qobj
    routes = {m.split("/")[1][:12]: dual["results"][m][i].get("route","?") for m in models}
    unique = set(routes.values())
    if len(unique) > 1:
        if not diff_found:
            print("\n\u26a1 Queries có kết quả khác nhau:")
            diff_found = True
        print(f"  \u2022 \"{q}\"  \u2192  {routes}")
if not diff_found:
    print("\n\u2705 Cả 3 model cho kết quả giống hệt nhau trên 20 queries.")
    print("   \u2192 Claim Verifier (\u00a711) là bottleneck chính, không phải model LLM.")
    print("   \u2192 Cải thiện Claim Verifier hoặc thêm LiveContext sẽ tăng success rate.")

In [ ]:
# Chart 7: 2x2 comparison
import numpy as np
n_models = len(models)
colors_m = ["#3b82f6", "#f59e0b", "#10b981"][:n_models]
short = [m.split("/")[1][:12] for m in models]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 7a: Success rate bar
ax = axes[0, 0]
ok_counts = []
for m in models:
    ok = len([r for r in dual["results"][m] if r.get("route") != "abstain" and "error" not in r])
    ok_counts.append(ok)
bars = ax.bar(short, [c/total_q for c in ok_counts], color=colors_m)
for b, c in zip(bars, ok_counts):
    ax.text(b.get_x()+b.get_width()/2., b.get_height()+0.02, f"{c}/{total_q}",
            ha="center", fontweight="bold", fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Tỉ lệ trả lời")
ax.set_title("Success Rate")

# 7b: Avg latency (OK vs Abstain)
ax = axes[0, 1]
x_pos = np.arange(n_models)
ok_avgs, ab_avgs = [], []
for m in models:
    ok_lat = [r["latency_ms"] for r in dual["results"][m] if r.get("route") != "abstain" and "error" not in r]
    ab_lat = [r["latency_ms"] for r in dual["results"][m] if r.get("route") == "abstain"]
    ok_avgs.append(sum(ok_lat)/len(ok_lat) if ok_lat else 0)
    ab_avgs.append(sum(ab_lat)/len(ab_lat) if ab_lat else 0)
w = 0.35
ax.bar(x_pos - w/2, ok_avgs, w, label="OK", color="#10b981")
ax.bar(x_pos + w/2, ab_avgs, w, label="Abstain", color="#ef4444")
for i in range(n_models):
    ax.text(i-w/2, ok_avgs[i]+100, f"{ok_avgs[i]:.0f}ms", ha="center", fontsize=8)
    ax.text(i+w/2, ab_avgs[i]+100, f"{ab_avgs[i]:.0f}ms", ha="center", fontsize=8)
ax.set_xticks(x_pos)
ax.set_xticklabels(short)
ax.set_ylabel("Avg Latency (ms)")
ax.set_title("Latency: OK vs Abstain")
ax.legend()

# 7c: Success by category
ax = axes[1, 0]
cats = []
for qobj in dual["queries"]:
    c = qobj.get("category", "?") if isinstance(qobj, dict) else "?"
    if c not in cats: cats.append(c)
cat_data = {m: {} for m in models}
for m in models:
    for i, qobj in enumerate(dual["queries"]):
        c = qobj.get("category", "?") if isinstance(qobj, dict) else "?"
        r = dual["results"][m][i]
        if c not in cat_data[m]: cat_data[m][c] = {"ok": 0, "total": 0}
        cat_data[m][c]["total"] += 1
        if r.get("route") != "abstain" and "error" not in r:
            cat_data[m][c]["ok"] += 1
x = np.arange(len(cats))
w3 = 0.25
for j, m in enumerate(models):
    rates = [cat_data[m].get(c, {"ok":0,"total":1})["ok"]/cat_data[m].get(c,{"ok":0,"total":1})["total"] for c in cats]
    ax.bar(x + (j-1)*w3, rates, w3, label=short[j], color=colors_m[j])
ax.set_xticks(x)
ax.set_xticklabels(cats, fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_title("Success by Category")
ax.legend(fontsize=8)

# 7d: DeepSeek wins
ax = axes[1, 1]
win_data = {"Chỉ DeepSeek OK": 0, "Tất cả OK": 0, "Tất cả Abstain": 0, "Khác": 0}
ds = models[-1] if "deepseek" in models[-1].lower() else models[0]
others = [m for m in models if m != ds]
for i in range(total_q):
    ds_ok = dual["results"][ds][i].get("route") != "abstain" and "error" not in dual["results"][ds][i]
    others_ok = all(dual["results"][m][i].get("route") != "abstain" and "error" not in dual["results"][m][i] for m in others)
    if ds_ok and not others_ok: win_data["Chỉ DeepSeek OK"] += 1
    elif ds_ok and others_ok: win_data["Tất cả OK"] += 1
    elif not ds_ok and not others_ok: win_data["Tất cả Abstain"] += 1
    else: win_data["Khác"] += 1
pie_colors = ["#10b981", "#3b82f6", "#ef4444", "#94a3b8"]
vals = [v for v in win_data.values() if v > 0]
lbls = [k for k, v in win_data.items() if v > 0]
ax.pie(vals, labels=lbls, colors=pie_colors[:len(vals)], autopct="%1.0f%%", startangle=90)
ax.set_title("DeepSeek vs GPT")
plt.tight_layout()
plt.show()

### 14.1 Phân tích kết quả

**Tại sao success rate chỉ 40-50%?** Phân tích theo category:

In [ ]:
# Phân tích theo category từ data thật
cats_order = ["KB FAQ", "Menu", "Allergy", "Order", "Off-topic", "Chitchat"]
abstain_reasons = {"KB FAQ": "Query không có trong 213 chunks KB",
    "Menu": "Claim Verifier chặn (không có evidence trong KB)",
    "Allergy": "Cần menu data + LLM reasoning",
    "Order": "Pipeline chưa có logic đặt hàng",
    "Off-topic": "Đúng — hệ thống không trả lời ngoài scope",
    "Chitchat": "Chưa handle câu xã giao"}
rows14 = []
for cat in cats_order:
    idxs = [i for i, qo in enumerate(dual["queries"]) if (qo.get("category","") if isinstance(qo,dict) else "") == cat]
    if not idxs: continue
    row = {"Category": cat, "Queries": len(idxs)}
    for m in models:
        ok = sum(1 for i in idxs if dual["results"][m][i].get("route") != "abstain" and "error" not in dual["results"][m][i])
        label = m.split("/")[1][:12]
        row[label] = f"{ok}/{len(idxs)} ({ok/len(idxs):.0%})"
    row["Nguyên nhân Abstain"] = abstain_reasons.get(cat, "")
    rows14.append(row)
display(pd.DataFrame(rows14).style.hide(axis="index").set_caption(
    "Phân tích success theo category"))

### 14.2 Giải thích kiến trúc: Tại sao nhiều query abstain?

Pipeline dùng thiết kế **fail-closed** — chỉ trả lời khi **chắc chắn đúng**:

```
Query → Guardrails → Intent Router → 3 đường:
  ├─ kb_rag:   Hybrid RRF tìm evidence → trả lời trực tiếp (không LLM)
  ├─ live_data: Lookup menu_items → trả giá trực tiếp (không LLM)
  └─ llm_gen:  LLM sinh response → Claim Verifier kiểm tra
                                     └─ Nếu không có evidence → ABSTAIN
```

**4 nguyên nhân chính cho abstain:**

| # | Nguyên nhân | Ví dụ | Giải pháp |
|---|---|---|---|
| 1 | **KB thiếu** | "phòng riêng", "hủy đơn" | Thêm vào KB |
| 2 | **Claim Verifier chặn** | "món không cay" | Verifier cần check LiveContext |
| 3 | **Order chưa hỗ trợ** | "thêm 1 trà đá" | Cần Cart API |
| 4 | **Off-topic đúng** | "thời tiết" | Không cần fix |

> **Quan trọng:** Abstain **không phải lỗi** — là thiết kế có chủ đích.
> Trong nhà hàng thật, khi chatbot abstain, hệ thống hiển thị
> câu mặc định như "Xin lỗi, tôi chưa có thông tin này" thay vì
> trả lời sai (hallucination).

### 14.3 So sánh 3 Model

In [ ]:
# Tính từ data, không hardcode
from scripts.notebook_metrics import is_strict_pipeline_success

comp_rows = []
for m in models:
    res = dual["results"][m]
    ok = [r for r in res if r.get("route") != "abstain" and "error" not in r]
    strict_n = sum(1 for r in res if "error" not in r and is_strict_pipeline_success(r))
    ab = [r for r in res if r.get("route") == "abstain"]
    # Tách fast-path vs LLM
    fast = [r for r in ok if r.get("route") in ("kb_rag","clarify","live_data")]
    llm = [r for r in ok if r.get("route") not in ("kb_rag","clarify","live_data","abstain")]
    fast_lat = [r["latency_ms"] for r in fast]
    llm_lat = [r["latency_ms"] for r in llm]
    ab_lat = [r["latency_ms"] for r in ab]
    # KB FAQ subset
    kb_idxs = [i for i, q in enumerate(dual["queries"]) if (q.get("category","") if isinstance(q,dict) else "") == "KB FAQ"]
    kb_ok = sum(1 for i in kb_idxs if res[i].get("route") != "abstain" and "error" not in res[i])
    # Menu+Allergy
    ma_idxs = [i for i, q in enumerate(dual["queries"]) if (q.get("category","") if isinstance(q,dict) else "") in ("Menu","Allergy")]
    ma_ok = sum(1 for i in ma_idxs if res[i].get("route") != "abstain" and "error" not in res[i])
    comp_rows.append({
        "Model": m.split("/")[1][:15],
        "Availability": f"{len(ok)}/{total_q} ({len(ok)/total_q:.0%})",
        "Strict": f"{strict_n}/{total_q} ({strict_n/total_q:.0%})",
        "KB FAQ": f"{kb_ok}/{len(kb_idxs)}",
        "Menu+Allergy": f"{ma_ok}/{len(ma_idxs)}",
        "Fast-path (ms)": f"{sum(fast_lat)/len(fast_lat):.0f}" if fast_lat else "-",
        "LLM path (ms)": f"{sum(llm_lat)/len(llm_lat):.0f}" if llm_lat else "-",
        "Abstain (ms)": f"{sum(ab_lat)/len(ab_lat):.0f}" if ab_lat else "-",
    })
display(pd.DataFrame(comp_rows).style.hide(axis="index").set_caption(
    f"So sánh 3 model trên {total_q} queries"))

In [ ]:
from IPython.display import Markdown, display
from scripts.notebook_metrics import summarize_dual_model, format_part4_narrative

_dual_summary = summarize_dual_model(dual)
display(Markdown(format_part4_narrative(_dual_summary)))

## 14.4 Thí nghiệm chọn kiến trúc production (cố định DeepSeek)

Đây là thí nghiệm quyết định production. Nó khác §14 ở chỗ **model được giữ cố định** là `oc/deepseek-v4-flash-free`; biến độc lập duy nhất là profile pipeline.

- Single-turn: KB, menu, giá, category, tag, gợi ý và ba câu production.
- Multi-turn: ordinal referent, loại món, đổi số người/chủ đề/sở thích, duy trì dị ứng.
- Safety: ID/giá giả, món ngoài menu, dị ứng, injection, session isolation.
- Availability: timeout/response lỗi; LLM case chạy ba lượt, deterministic case một lượt.

Thứ tự chọn: **safety hard gate → strict semantic quality → context accuracy → p95 latency → số lượt gọi DeepSeek**. Không có profile qua hard gate thì deployment bị chặn.

In [ ]:
from scripts.notebook_metrics import summarize_pipeline_selection

pipeline_selection_path = AI_ROOT / "evaluation" / "results" / "pipeline_selection.json"
if pipeline_selection_path.exists():
    pipeline_selection = json.loads(pipeline_selection_path.read_text(encoding="utf-8"))
else:
    pipeline_selection = {
        "winner": None,
        "selection_reason": "artifact_not_generated",
        "profiles": [],
    }
pipeline_selection_summary = summarize_pipeline_selection(pipeline_selection)
selection_rows = pipeline_selection_summary["rows"]
if selection_rows:
    display(pd.DataFrame(selection_rows).style.hide(axis="index").format({
        "strict_semantic_success": "{:.1%}",
        "context_accuracy": "{:.1%}",
        "p95_latency_ms": "{:.1f}",
        "mean_llm_calls": "{:.2f}",
        "run_to_run_disagreement_rate": "{:.1%}",
    }).set_caption("Controlled DeepSeek pipeline comparison"))
else:
    print("Chưa có pipeline_selection.json — chạy run_pipeline_profile_eval.py; production vẫn BLOCKED.")
print("Winner:", pipeline_selection_summary["winner"])
print("Commit:", pipeline_selection_summary["commit_sha"])
print("Dataset:", pipeline_selection_summary["dataset_hash"])

In [ ]:
import numpy as np

if selection_rows:
    labels = [row["profile"] for row in selection_rows]
    strict = [100 * row["strict_semantic_success"] for row in selection_rows]
    context = [100 * row["context_accuracy"] for row in selection_rows]
    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(x - 0.18, strict, 0.36, label="Strict semantic success")
    ax.bar(x + 0.18, context, 0.36, label="Context accuracy")
    ax.set_xticks(x, labels)
    ax.set_ylim(0, 100)
    ax.set_ylabel("%")
    ax.set_title("So sánh profile sau safety gate")
    ax.legend()
    plt.tight_layout()
    plt.show()

---
# PHẦN V — KẾT LUẬN

## 15. Kết luận lựa chọn kiến trúc production

Kết luận dưới đây được sinh trực tiếp từ `pipeline_selection.json`, không hardcode winner. Production chỉ được bật profile thắng trên đúng model và commit đã đánh giá.

In [ ]:
from scripts.notebook_metrics import format_pipeline_selection_conclusion

display(Markdown(format_pipeline_selection_conclusion(pipeline_selection)))

## 15.1 Tổng hợp kết quả 5 phần

In [ ]:
# Tổng hợp
ir=live_test["intent_results"]; gr=live_test["guard_results"]
cr=live_test["claim_results"]; pr=live_test["pipeline_results"]
mr=live_test["multi_results"]

i_ok=len([r for r in ir if r["intent"]!="general"])
g_ok=len(gr); c_ok=len(cr)
p_ok=len([r for r in pr if "error" not in r and r.get("route")!="abstain"])
m_ok=len([r for r in mr if "error" not in r and r.get("route")!="abstain"])

dual_summary_rows = []
if "dual" in globals():
    for m in dual["models"]:
        ok_m = len([r for r in dual["results"][m] if r.get("route") != "abstain" and "error" not in r])
        dual_summary_rows.append((m.split("/")[-1][:20], ok_m, len(dual["queries"])))
elif (AI_ROOT / "evaluation" / "results" / "dual_model_test.json").exists():
    _d = json.load(open(AI_ROOT / "evaluation" / "results" / "dual_model_test.json", encoding="utf-8"))
    for m in _d["models"]:
        ok_m = len([r for r in _d["results"][m] if r.get("route") != "abstain" and "error" not in r])
        dual_summary_rows.append((m.split("/")[-1][:20], ok_m, len(_d["queries"])))
dual_label = ", ".join(f"{n} {o}/{t}" for n, o, t in dual_summary_rows) if dual_summary_rows else "xem §14"
dual_avg = (
    sum(o / t for _, o, t in dual_summary_rows) / len(dual_summary_rows)
    if dual_summary_rows else p_ok / len(pr)
)

summary=[
  {"Part":"I","Thành phần":"Knowledge Base","Kết quả":"26 files, 213 chunks",
   "Vai trò":"Nguồn evidence cho retrieval + claim verify"},
  {"Part":"II","Thành phần":"Hybrid RRF","Kết quả":hit5_label,
   "Vai trò":"Tìm chunk phù hợp cho LLM"},
  {"Part":"III","Thành phần":"Intent Router","Kết quả":f"{i_ok}/{len(ir)} ({i_ok/len(ir):.0%})",
   "Vai trò":"Quyết định dùng Hybrid RRF hay Catalog"},
  {"Part":"III","Thành phần":"Guardrails","Kết quả":f"{g_ok}/{len(gr)} (100%)",
   "Vai trò":"Chặn PII, injection, off-topic"},
  {"Part":"III","Thành phần":"Claim Verifier","Kết quả":f"{c_ok}/{len(cr)} (100%)",
   "Vai trò":"Chặn hallucination (dùng KB Part I)"},
  {"Part":"IV","Thành phần":"Pipeline (LLM)","Kết quả":f"{p_ok}/{len(pr)} ({p_ok/len(pr):.0%})",
   "Vai trò":"End-to-end với " + live_test["model"]},
  {"Part":"IV","Thành phần":"Multi-turn","Kết quả":f"{m_ok}/{len(mr)} ({m_ok/len(mr):.0%})",
   "Vai trò":"Context retention qua lượt"},
  {"Part":"IV","Thành phần":"So sánh 3 model","Kết quả":dual_label,
   "Vai trò":"Non-abstain trên 20 query (dual_model_test.json)"},
]
display(pd.DataFrame(summary).style.hide(axis="index").set_caption("Tổng hợp 5 phần"))

In [ ]:
# Chart 7: Radar chart
import numpy as np
labels = ["Intent\nRouter", "Guardrails", "Claim\nVerifier", "Pipeline\n(LLM)", "Multi\nturn"]
scores = [i_ok/len(ir), g_ok/len(gr), c_ok/len(cr), p_ok/len(pr), m_ok/len(mr)]

angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
scores_r = scores + [scores[0]]
angles_r = angles + [angles[0]]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.fill(angles_r, scores_r, color="#3b82f6", alpha=0.25)
ax.plot(angles_r, scores_r, color="#3b82f6", linewidth=2)
ax.scatter(angles, scores, color="#3b82f6", s=60, zorder=5)
for angle, score, label in zip(angles, scores, labels):
    ax.text(angle, score+0.08, f"{score:.0%}", ha="center", fontsize=10, fontweight="bold")
ax.set_xticks(angles)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_title("Component Scores (chạy thật)", pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 8: Summary bar
fig, ax = plt.subplots(figsize=(10, 4))
comp_names = ["KB\n(Part I)", "Hybrid RRF\n(Part II)", "Intent\n(Part III)",
              "Guardrails\n(Part III)", "Claim\n(Part III)", "Pipeline\n(Part IV)",
              "Multi-turn\n(Part IV)", "3 model\n(Part IV)"]
comp_scores = [1.0, hit5_score, i_ok/len(ir), g_ok/len(gr), c_ok/len(cr), p_ok/len(pr), m_ok/len(mr), dual_avg]
comp_colors = ["#6366f1","#8b5cf6","#f59e0b","#ef4444","#ec4899","#3b82f6","#10b981","#0ea5e9"]

bars = ax.bar(comp_names, comp_scores, color=comp_colors)
for b, v in zip(bars, comp_scores):
    ax.text(b.get_x()+b.get_width()/2., b.get_height()+0.02,
            f"{v:.0%}", ha="center", fontsize=10, fontweight="bold")
ax.set_ylim(0, 1.15)
ax.axhline(y=1.0, color="#e5e7eb", linestyle="--", linewidth=0.8)
ax.set_title("Tổng hợp: Part I → Part IV (chạy thật)")
plt.figtext(0.5, 0.01, "Cột Hybrid RRF = Hit@5 screening Part II (107 case), không phải release gate 110 case.",
            ha="center", fontsize=9, style="italic")
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)
plt.show()

### Nhận xét tổng hợp

> **Lưu ý về metric:** Guardrails và Claim Verifier đạt 100%
> nghĩa là **detect rate** (phát hiện đúng mọi test case), không phải
> precision trên traffic thật. Cần eval production để biết false-positive.

**Mạnh (>= 80%):**
- **Retrieval** (Part II): Hybrid RRF — Hit@5 từ `notebook_retrieval_screening.json`
- **Guardrails** + **Claim Verifier**: 100% detect rate (Part III)
- **Multi-turn** (§13): retention trên bộ turn thiết kế

**Yếu (< 80%) / cần tách metric:**
- **Intent Router**: miss teencode và edge cases
- **Pipeline §12** vs **so sánh 3 model §14**: cùng 20 query, metric/timestamp khác nhau
- **Artifact lịch sử Part IV**: nhiều abstain do evidence assembly khi đó chưa có LiveContext

**Cách đọc đúng kết quả:**
Lỗi evidence assembly đã được sửa chung trước thí nghiệm profile ở §14.4. Vì vậy artifact lịch sử giải thích sự cố cũ, còn `pipeline_selection.json` mới là nguồn quyết định kiến trúc production.

## 16. Hạn chế và hướng phát triển

### Hạn chế

| Hạn chế | Ảnh hưởng | Mức độ |
|---|---|---|
| Dữ liệu tự xây (KB, eval set) | Kết quả không generalize | Cao |
| Sample size nhỏ (20 intent / 20 pipeline / 20×3 model) | Không thống kê mạnh | Cao |
| `route` null trong export JSON | Availability đếm optimistic | Cao |
| Evidence assembly KB/menu | Đã sửa chung; vẫn cần golden regression để tránh tái phát | Cao |
| Chưa có traffic thật | Chưa biết production quality | Cao |
| 3-model chưa khác biệt ở KB path | KB fast-path không dùng LLM | TB |
| Intent rule-based miss teencode | Miss queries Vietnamese noisy | TB |
| Dense E5 yếu noisy | Cần normalize (Part II) | TB |
| Claim Verifier lexical | False-reject paraphrase | Thấp |

### Bản đồ bằng chứng (staging)

| Nguồn trong notebook | Artifact / doc | Ý nghĩa release |
|---|---|---|
| Retrieval Part II | `notebook_retrieval_screening.json` | Hit@5 screening trong notebook (sau execute Part II) |
| Retrieval release | `dev_retrieval_summary.v3.json` (nếu có) | Gate staging Hit@5 — khác bộ case Part II |
| Session / multi-turn | `evaluation/results/session_e2e_eval.json` | Context retention offline |
| Golden LLM eval | `evaluation/results/golden_llm_eval_*.json` | Baseline lịch sử; không headline 100% |
| Paired GPT/DeepSeek quality | `evaluation/dual_model/.../comparison.json` | Protocol PASS; quality CHƯA PASS release |
| Live / dual notebook | `notebook_live_test.json`, `dual_model_test.json` | Demo pipeline; không thay human eval |
| Trạng thái tổng | [`docs/ai/AI_STAGING_READINESS.md`](../../docs/ai/AI_STAGING_READINESS.md) | **NOT READY** cho đến khi đủ gate |

### Hướng phát triển

| Ưu tiên | Hướng | Impact |
|---|---|---|
| **CAO** | Test với menu data thật (LiveContext) | Mở khóa menu + dual-model so sánh |
| **CAO** | Eval từ traffic thật + human eval | Kết quả sát thực tế |
| **CAO** | Thêm teencode rules cho Intent | Vượt 75% accuracy |
| TB | ML-based intent (BERT/PhoBERT) | Bắt paraphrase |
| TB | Fine-tune E5 Vietnamese | Cải thiện noisy retrieval |
| Thấp | Cross-encoder rerank | Hit@1 tốt hơn |

## 17. Kết luận

Notebook này đã **chạy thật** và đánh giá toàn bộ pipeline RAG chatbot:

1. **Part I** — 26 files KB → 213 chunks. Normalize tiếng Việt giúp BM25 +49% noisy.
2. **Part II** — So sánh 3 methods: **Hybrid RRF tốt nhất** trên bộ screening (`notebook_retrieval_screening.json`).
   BM25 mạnh noisy, Dense mạnh clean → kết hợp là tối ưu.
3. **Part III** — 4 thành phần pipeline:
   - Intent Router chọn **khi nào dùng Hybrid RRF** (Part II)
   - Guardrails chặn queries nguy hiểm **trước** khi tốn chi phí LLM
   - Session Memory nhớ dị ứng, context qua **nhiều lượt**
   - Claim Verifier dùng **KB + LiveContext menu** làm evidence chống hallucination

In [ ]:
from scripts.notebook_metrics import format_part17_bullet_part4, summarize_dual_model

if "dual" not in globals():
    dual = json.load(open(
        AI_ROOT / "evaluation" / "results" / "dual_model_test.json", encoding="utf-8"
    ))
print(format_part17_bullet_part4(summarize_dual_model(dual)))
print()
print("**Tính năng product:** Dữ liệu thật (LiveContext), Cart Suggestion,")
print("refresh không mất chat, session theo phiên bàn.")
print()
print("> Hệ thống chạy được trên **VPS 4vCPU/8GB, không GPU**.")
print("> Production chỉ chạy winner trong pipeline_selection.json; tiếp tục human eval và staging load test.")

---

## 18. Đưa vào production — kết luận báo cáo

Phần này **chốt báo cáo**: kết quả nghiên cứu trong notebook đã được **ứng dụng** vào chatbot AI hiện tại (Python AI service + backend .NET trên staging/production) ở mức nào, và **stack nào** nhóm cam kết vận hành.

### Tính năng từ notebook → hệ thống đang chạy

| Nội dung nghiên cứu (notebook) | Trạng thái trên hệ thống | Ghi chú triển khai |
|---|---|---|
| **Knowledge Base** Part I — 26 file MD, chunk theo `##`, question variants | **Đã áp dụng** | `knowledge-base/` load lúc khởi động AI service |
| **Chuẩn hóa tiếng Việt** (teencode, không dấu, emoji) | **Đã áp dụng** | `vietnamese_normalizer` trong BM25 và pipeline query |
| **Retrieval Hybrid RRF** (BM25 + Dense E5 small) Part II | **Đã áp dụng** | `RAG_RETRIEVAL_METHOD=hybrid`, `AI_EMBEDDING_MODEL=e5_small` — ADR retriever |
| **Ba profile pipeline** Part III | **Đã áp dụng để thử nghiệm** | `llm_first_v1`, `evidence_first_v2`, `planner_state_v3`; production chỉ bật winner |
| **Guardrails** — PII, prompt injection, chống tự đặt món / bịa giá | **Đã áp dụng** | `guardrails.detect_guardrail_flags`; injection **chặn trước LLM** (`assistant.py`) |
| **Chặn / xử lý câu hỏi sai chủ đề** Part III | **Đã áp dụng một phần** | Cờ `OUT_OF_SCOPE` + chunk KB `out-of-domain-redirect`; LLM + prompt hướng dẫn từ chối — **chưa** có nhánh từ chối cứng cho mọi off-topic như injection |
| **Ngữ cảnh hội thoại** (session, rolling summary, lịch sử) Part III | **Đã áp dụng** | Backend gửi `session_memory`, `rolling_summary`, `session_state`, history → `rewrite_query` và prompt LLM |
| **Claim Verifier** chống bịa sau LLM | **Đã áp dụng (KB + LiveContext)** | `verify_claims` trên chunk KB và live menu IDs; evidence assembly dùng chung cho ba profile |
| **Structured response** (evidence, claims, guardrail_flags, cart gợi ý) | **Đã áp dụng** | Contract `/v1/chat` — backend kiểm tra trước khi hiển thị |
| **LLM DeepSeek** qua 9router | **Đã áp dụng (staging/production)** | `LLM_MODEL=oc/deepseek-v4-flash-free` |
| So sánh **3 model** / metric Part IV | **Không đưa vào production** | Chỉ phục vụ thí nghiệm và báo cáo |
| Human eval, gate chất lượng end-to-end | **Chưa hoàn tất release** | [`AI_STAGING_READINESS.md`](../../docs/ai/AI_STAGING_READINESS.md) — **NOT READY**; ưu tiên LLM-first + eval golden sau thay đổi routing |

### Stack production nhóm chốt vận hành

| Thành phần | Giá trị |
|---|---|
| Chat routing | `AI_PIPELINE_PROFILE=<winner từ pipeline_selection.json>` |
| Retrieval | `hybrid` + `e5_small` |
| LLM duy nhất | `oc/deepseek-v4-flash-free` (9router), không GPT fallback |
| Tích hợp | `CHAT_AI_PROVIDER=python-rag`, Docker [`deploy/docker-compose.yml`](../../deploy/docker-compose.yml) |
| Kiểm tra sau deploy | [`VPS_STAGING_AI_RUNBOOK.md`](../../docs/ai/VPS_STAGING_AI_RUNBOOK.md) |

### Kết luận báo cáo

Notebook chứng minh **phương pháp** (RAG hybrid, guardrails, routing, session) và **đo** trên bộ eval tự xây; **hệ thống thật** đã gắn cùng module code với cấu hình trên. Evidence assembly KB + LiveContext đã được sửa chung trước thí nghiệm profile. Production chỉ chạy winner sau safety gate; phần còn lại là human review và staging load test.